# Evaluation

This notebook is the central readout for LipSynth.

**Simple boundary:** `measure how good Stage 1 and Stage 2 are`

It reads existing artifacts and shows:

- Stage 1 benchmark summaries and confidence/error analysis
- Stage 2 metrics, ablations, and output quality summaries
- joined per-clip analysis with Stage 1 predicted text and Stage 2 generated-audio transcripts
- a final interpretation of where the pipeline is failing or improving


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Audio, display

try:
    import whisper
except ImportError:
    whisper = None


def find_pipeline_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / 'pyproject.toml').exists() and (cand / 'scripts').exists():
            return cand
    raise FileNotFoundError('Could not locate Pipeline root')


def choose_stage2_dir(pipe_root: Path) -> Path:
    candidates = [
        pipe_root / 'outputs' / 'stage2_finetune',
        pipe_root / 'outputs' / 'stage2_finetune_smoke2',
        pipe_root / 'outputs' / 'stage2_finetune_smoke',
    ]
    for candidate in candidates:
        if (candidate / 'final_report.json').exists() or (candidate / 'ablation_guidance.csv').exists():
            return candidate
    return candidates[0]


PIPE_ROOT = find_pipeline_root(Path.cwd())
STAGE1_DIR = PIPE_ROOT / 'outputs' / 'stage1_eval'
STAGE1_PRETRAINED = STAGE1_DIR / 'pretrained'
STAGE1_RETRAIN = STAGE1_DIR / 'retrain'
STAGE2_DIR = choose_stage2_dir(PIPE_ROOT)
AUDIO_DIR = STAGE2_DIR / 'audio_samples'

print('PIPE_ROOT =', PIPE_ROOT)
print('STAGE1_DIR =', STAGE1_DIR)
print('STAGE2_DIR =', STAGE2_DIR)


## 1. Stage 1 Benchmark Summary

Load Stage 1 artifacts and summarize pretrained vs retrained performance when available.


In [ ]:
def load_json(path: Path):
    return json.loads(path.read_text()) if path.exists() else None

summary_rows = []
for label, path in [
    ('pretrained_val', STAGE1_PRETRAINED / 'val_summary_stage1_pretrained.json'),
    ('retrain_val', STAGE1_RETRAIN / 'val_summary_stage1_retrain.json'),
    ('retrain_eval', STAGE1_RETRAIN / 'val_summary_stage1_retrain_eval.json'),
    ('retrain_smoke', STAGE1_DIR / 'retrain_smoke' / 'val_summary_stage1_retrain.json'),
]:
    payload = load_json(path)
    if payload is None:
        continue
    summary_rows.append({
        'source': label,
        'path': str(path),
        'overall_wer': payload.get('overall_wer'),
        'overall_cer': payload.get('overall_cer'),
        'decision': payload.get('decision'),
        'checkpoint': payload.get('best_checkpoint') or payload.get('outputs', {}).get('checkpoint'),
    })

stage1_summary_df = pd.DataFrame(summary_rows)
display(stage1_summary_df if not stage1_summary_df.empty else pd.DataFrame([{'note': 'No Stage 1 summary artifacts found'}]))

stage1_pred_candidates = [
    STAGE1_PRETRAINED / 'val_predictions_stage1_pretrained.csv',
    STAGE1_RETRAIN / 'val_predictions_stage1_retrain.csv',
    STAGE1_DIR / 'val_predictions_stage1.csv',
]
stage1_pred_path = next((path for path in stage1_pred_candidates if path.exists()), None)
stage1_pred_df = pd.read_csv(stage1_pred_path) if stage1_pred_path else pd.DataFrame()
if not stage1_pred_df.empty:
    cols = [c for c in ['clip_id', 'gt_text', 'pred_text', 'sequence_confidence', 'wer', 'cer'] if c in stage1_pred_df.columns]
    display(stage1_pred_df[cols].head(10))


In [ ]:
error_bucket_candidates = [
    STAGE1_DIR / 'stage1_error_buckets.csv',
    STAGE1_PRETRAINED / 'val_errors_stage1_pretrained.json',
]
error_bucket_path = next((path for path in error_bucket_candidates if path.exists() and path.suffix == '.csv'), None)
if error_bucket_path is not None:
    error_buckets_df = pd.read_csv(error_bucket_path)
    print('Stage 1 error buckets:')
    display(error_buckets_df.head(20))
elif (STAGE1_PRETRAINED / 'val_errors_stage1_pretrained.json').exists():
    error_payload = load_json(STAGE1_PRETRAINED / 'val_errors_stage1_pretrained.json')
    print('Stage 1 error analysis JSON preview:')
    display(pd.DataFrame(error_payload.get('highest_wer_examples', [])[:10]))
else:
    print('No Stage 1 error bucket artifact found.')

if not stage1_pred_df.empty and 'sequence_confidence' in stage1_pred_df.columns:
    ax = stage1_pred_df['sequence_confidence'].hist(bins=20, figsize=(6, 4))
    ax.set_title('Stage 1 confidence distribution')
    ax.set_xlabel('sequence_confidence')
    plt.show()


## 2. Stage 2 Benchmark Summary

Load the reproducible Stage 2 evaluation outputs and summarize quality metrics and guidance ablations.


In [ ]:
final_report_path = STAGE2_DIR / 'final_report.json'
full_metrics_path = STAGE2_DIR / 'test_full_metrics.csv'
guidance_path = STAGE2_DIR / 'ablation_guidance.csv'
conf_guidance_path = STAGE2_DIR / 'ablation_conf_weighted.csv'
failure_buckets_path = STAGE2_DIR / 'failure_buckets.csv'
summary_path = STAGE2_DIR / 'evaluation_summary.csv'

final_report = load_json(final_report_path) if final_report_path.exists() else None
full_metrics_df = pd.read_csv(full_metrics_path) if full_metrics_path.exists() else pd.DataFrame()
guidance_df = pd.read_csv(guidance_path) if guidance_path.exists() else pd.DataFrame()
conf_guidance_df = pd.read_csv(conf_guidance_path) if conf_guidance_path.exists() else pd.DataFrame()
summary_df = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()

if final_report is not None:
    stage2_report_row = {
        'checkpoint_source': final_report.get('stage2', {}).get('checkpoint', {}).get('source'),
        'checkpoint_step': final_report.get('stage2', {}).get('checkpoint', {}).get('step'),
        'full_stoi_mean': final_report.get('stage2', {}).get('full_metrics', {}).get('stoi_mean'),
        'full_pesq_mean': final_report.get('stage2', {}).get('full_metrics', {}).get('pesq_mean'),
        'best_fixed_weight': final_report.get('guidance', {}).get('best_fixed_weight'),
    }
    display(pd.DataFrame([stage2_report_row]))
else:
    print('Missing final report:', final_report_path)

if not guidance_df.empty:
    display(guidance_df)
    ax = guidance_df.plot(x='w_video', y=['stoi_mean', 'pesq_mean'], marker='o', figsize=(7, 4), title='Stage 2 guidance ablation')
    ax.set_xlabel('w_video')
    plt.show()
else:
    print('Missing guidance ablation CSV:', guidance_path)

if not conf_guidance_df.empty:
    print('Confidence-weighted guidance:')
    display(conf_guidance_df.head())
if failure_buckets_path.exists():
    display(pd.read_csv(failure_buckets_path))
if not summary_df.empty:
    print('Evaluation summary rows:')
    display(summary_df)


## 3. Joined Per-Clip Analysis

Join Stage 1 predictions with Stage 2 audio metrics, then add optional ASR transcripts from generated audio.


In [ ]:
joined_df = pd.DataFrame()
if not stage1_pred_df.empty and not full_metrics_df.empty:
    left_cols = [c for c in ['clip_id', 'pred_text', 'sequence_confidence', 'wer', 'cer'] if c in stage1_pred_df.columns]
    right_cols = [c for c in ['clip_id', 'gt_text', 'stoi', 'pesq'] if c in full_metrics_df.columns]
    joined_df = full_metrics_df[right_cols].merge(stage1_pred_df[left_cols], on='clip_id', how='left')
    display(joined_df.head(20))
else:
    print('Need both Stage 1 predictions and Stage 2 full metrics for the joined view.')


In [ ]:
_whisper_model = None

def get_whisper_model():
    global _whisper_model
    if whisper is None:
        return None
    if _whisper_model is None:
        _whisper_model = whisper.load_model('base')
    return _whisper_model

transcript_rows = []
if not joined_df.empty:
    for clip_id in joined_df['clip_id'].head(3):
        wav_candidates = [
            AUDIO_DIR / f'{clip_id}_gen_full.wav',
            AUDIO_DIR / f'{clip_id}_generated.wav',
        ]
        wav_path = next((path for path in wav_candidates if path.exists()), None)
        asr_text = None
        if wav_path and whisper is not None:
            model = get_whisper_model()
            result = model.transcribe(str(wav_path), language='en', fp16=False)
            asr_text = str(result['text']).strip().lower()
        transcript_rows.append({
            'clip_id': clip_id,
            'generated_wav': str(wav_path) if wav_path else None,
            'generated_asr_text': asr_text,
        })

transcript_df = pd.DataFrame(transcript_rows)
if whisper is None:
    print('Whisper is not installed. Install it for Stage 2 transcript analysis: uv pip install openai-whisper')
if not transcript_df.empty:
    joined_asr_df = joined_df.merge(transcript_df, on='clip_id', how='left')
    display(joined_asr_df.head(10))
    for row in joined_asr_df.head(3).to_dict(orient='records'):
        if row.get('generated_wav'):
            print(f"\nClip: {row['clip_id']}")
            print('GT text       :', row.get('gt_text'))
            print('Stage1 text   :', row.get('pred_text'))
            print('Stage2 ASR    :', row.get('generated_asr_text'))
            display(Audio(row['generated_wav']))


## 4. Final Interpretation

Use the artifact summaries to judge whether the pipeline is improving or failing.


In [ ]:
notes = []
if not stage1_summary_df.empty:
    best_stage1 = stage1_summary_df.sort_values('overall_wer').iloc[0]
    notes.append(f"Best Stage 1 WER: {best_stage1['overall_wer']:.4f} from {best_stage1['source']}")
    if best_stage1['overall_wer'] > 0.60:
        notes.append('Stage 1 is still weak enough that downstream text guidance should be treated carefully.')
    else:
        notes.append('Stage 1 looks usable for downstream guidance and inspection.')
if final_report is not None:
    full_metrics = final_report.get('stage2', {}).get('full_metrics', {})
    stoi = full_metrics.get('stoi_mean')
    pesq = full_metrics.get('pesq_mean')
    notes.append(f"Stage 2 full STOI mean: {stoi}")
    notes.append(f"Stage 2 full PESQ mean: {pesq}")
    if stoi is not None and stoi < 0.3:
        notes.append('Generated speech intelligibility is still poor.')
    if final_report.get('guidance', {}).get('best_fixed_weight') is not None:
        notes.append(f"Best fixed guidance weight so far: {final_report['guidance']['best_fixed_weight']}")

for note in notes:
    print('-', note)
